# Quickstart

Autor: Luis M. de la Cruz


In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import flopy
import xmf6

In [2]:
# <>----- Datos de la simulación -----<>
init = {
    'sim_name' : "quick_start",
    'exe_name' : "C:\\Users\\luiggi\\Documents\\GitSites\\xmf6\\mf6\\windows\\mf6",
#    'exe_name' : "../../mf6/macosarm/mf6",
    'sim_ws' : "output_qs"
}

# <>----- Componentes -----<>

## [TDIS] Discretización temporal 
time = {}

## [IMS] Solución numérica 
ims = {}

## [GWF] Modelo de flujo 
gwf = { 
    'modelname': init["sim_name"],
    'save_flows': True
}

# <>----- Paquetes -----<>

## [DIS] Discretización espacial 
Lx = 10
Ly = 10
nx = 20 # ncol
ny = 20 # nrow
dx = Lx / nx # delr
dy = Ly / ny # delc

dis = {
    'nrow': ny, 
    'ncol': nx,
    'delr': dx,
    'delc': dy,
}

## [IC] Condiciones iniciales 
ic = {}

## [CHD] Condiciones de frontera 
chd = {
    'stress_period_data': [[(0, 0, 0), 1.], 
                           [(0, dis['nrow']-1, dis['ncol']-1), 0.]]
}

## [NPF] Propiedades de flujo 
npf = {
    'save_specific_discharge': True,
}

## [OC] Configuración de la salida
oc = {
    'budget_filerecord': f"{init['sim_name']}.bud",
    'head_filerecord': f"{init['sim_name']}.hds",
    'saverecord': [("HEAD", "ALL"), ("BUDGET", "ALL")],
}

# <>----- Inicialización de las componentes -----<>
o_sim = xmf6.gwf.initialize(silent = True, init = init, time = time, ims = ims)

# <>----- Inicialización del modelo de flujo -----<>
o_gwf = xmf6.gwf.build(o_sim, silent = True, gwf = gwf, dis = dis, ic = ic, chd = chd, npf = npf, oc = oc)

# <>----- Escritura de los archivos de entrada para MODFLOW 6 -----<>
o_sim.write_simulation(silent = True)

# <>----- Ejecución de la simulación -----<>
o_sim.run_simulation(silent = True)

(True, [])

In [5]:
xmf6.nice_print(init, "Inicialización")


Inicialización
――――――――――――――
            sim_name = quick_start
            exe_name = C:\Users\luiggi\Documents\GitSites\xmf6\mf6\windows\mf6
              sim_ws = output_qs 
――――――――――――――


In [ ]:
# <>----- Recuperamos los resultados de la simulación -----<>
head = xmf6.gwf.get_head(o_sim, o_gwf)
qx, qy, qz, n_q = xmf6.gwf.get_specific_discharge(o_sim, o_gwf)
grid = o_gwf.modelgrid
x, y, z = grid.xyzcellcenters

In [ ]:
# <>----- Definición de la figura -----<>
fig, ax = plt.subplots(1,1, figsize=(6,6))
ax.set_aspect('equal')
pmv0 = flopy.plot.PlotMapView(model = o_gwf, ax = ax)
pmv0.plot_grid(colors = 'w', lw = 0.25, ls="--")
cb = pmv0.plot_array(head, cmap = "winter", alpha=0.5)
pmv0.plot_vector(qx, qy, normalize=False, color="darkblue", alpha=0.55)
ct = pmv0.contour_array(head, levels=20, colors='k', linewidths=1.5)
ax.clabel(ct, fontsize=14)
ax.streamplot(x, y[::-1][:], qx[0], qy[0][::-1], color='navy', 
              integration_direction = 'both', 
              density = [0.5, 0.5], linewidth = 0.75, broken_streamlines = True, 
              arrowstyle = "->", arrowsize = 0.75,  )
plt.colorbar(cb, ax = ax, label = "$h$ (m)", 
             cax = xmf6.vis.cax(ax, cb))
plt.show()

In [6]:
print(init)

{'sim_name': 'quick_start', 'exe_name': 'C:\\Users\\luiggi\\Documents\\GitSites\\xmf6\\mf6\\windows\\mf6', 'sim_ws': 'output_qs'}


In [7]:
for i in init.keys():
    print(i)

sim_name
exe_name
sim_ws


In [9]:
print(init.keys())
[len(k) for k in init.keys()]

dict_keys(['sim_name', 'exe_name', 'sim_ws'])


[8, 8, 6]

In [10]:
from functools import reduce

max_len = reduce(max, [len(k) for k in init.keys()])
print(max_len)

8


In [223]:
from colorama import Fore, Style, Back

def nice_print(data, message = ''):
    size = len(message)
    fmt1 = '{:^'+str(size)+'}'
    
    max_len = reduce(max, [len(k) for k in data.keys()])
    fmt2 = '{:>' + str(max_len) + '} = {}'

    print(Fore.BLUE)
    print(message)
    print(fmt1.format(size * chr(0x2015)) + Style.RESET_ALL)
    for k,v in data.items():
        if isinstance(v, list):
            print(fmt2.format(k, Fore.BLUE + chr(0x2015) + chr(0x2015) + " data array " \
                              + chr(0x2015) + chr(0x2015) + Style.RESET_ALL ))
            for l in v:
                print(fmt2.format("  ", l))
        else:
            print(fmt2.format(k, v))
    print(Fore.BLUE + fmt1.format(size * chr(0x2015)) + Style.RESET_ALL)
    

In [224]:
nice_print(init, "Inicia esta parte")


Inicia esta parte
―――――――――――――――――
sim_name = quick_start
exe_name = C:\Users\luiggi\Documents\GitSites\xmf6\mf6\windows\mf6
  sim_ws = output_qs
―――――――――――――――――


In [225]:
nice_print(dis, "dis espacial")


dis espacial
――――――――――――
nrow = 20
ncol = 20
delr = 0.5
delc = 0.5
――――――――――――


In [226]:
nice_print(chd, "boundary conditions")


boundary conditions
―――――――――――――――――――
stress_period_data = ―― data array ――
                   = [(0, 0, 0), 1.0]
                   = [(0, 19, 19), 0.0]
―――――――――――――――――――


In [227]:
nice_print(oc, "boundary conditions")


boundary conditions
―――――――――――――――――――
budget_filerecord = quick_start.bud
  head_filerecord = quick_start.hds
       saverecord = ―― data array ――
                  = ('HEAD', 'ALL')
                  = ('BUDGET', 'ALL')
―――――――――――――――――――
